# Messages Decoder
Decrypts `Data/EncodedData/messages_l2.enc` in two stages:
1. **Level 2** — decrypt the whole file (required)
2. **Level 1** — restore original sender names (optional, press Enter to skip)

In [18]:
import json
import base64
import hashlib
import getpass
from pathlib import Path
from cryptography.fernet import Fernet, InvalidToken

In [19]:
def derive_key(password: str, salt: bytes) -> Fernet:
    key_bytes = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, iterations=200_000)
    return Fernet(base64.urlsafe_b64encode(key_bytes))

def replace_all(obj, mapping):
    """Replace ALL occurrences of every token within any string (longest-first)."""
    if isinstance(obj, str):
        result = obj
        for src, dst in mapping:
            result = result.replace(src, dst)
        return result
    if isinstance(obj, list):
        return [replace_all(item, mapping) for item in obj]
    if isinstance(obj, dict):
        return {k: replace_all(v, mapping) for k, v in obj.items()}
    return obj

In [20]:
import os
base_dir = Path(os.getcwd()).parent
enc_path = base_dir / 'Data' / 'EncodedData' / 'messages_l2.enc'

if not enc_path.exists():
    raise FileNotFoundError(f"Encrypted file not found: {enc_path}")
print(f"Found: {enc_path}  ({enc_path.stat().st_size / 1024:.1f} KB)")

Found: /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/EncodedData/messages_l2.enc  (27234.3 KB)


## Step 1 — Decrypt the file (Level 2)

In [21]:
pwd2 = getpass.getpass('Level-2 password (file decryption): ')

try:
    fernet2 = derive_key(pwd2, b'philosophy_group_salt_l2')
    decrypted_bytes = fernet2.decrypt(enc_path.read_bytes())
    data = json.loads(decrypted_bytes.decode('utf-8'))
    print(f"Level-2 OK — {len(data['messages'])} messages, {len(data['participants'])} participants.")
except InvalidToken:
    raise ValueError("Wrong Level-2 password — cannot decrypt file.")

Level-2 OK — 44422 messages, 37 participants.


## Step 2 — Restore sender names (Level 1, optional)
Press **Enter** with no input to skip name decryption.

In [22]:
pwd1 = getpass.getpass('Level-1 password (name decryption, Enter to skip): ')

if pwd1.strip() == '':
    decoded = data
    print('Name decryption skipped — sender names remain as encrypted tokens.')
else:
    try:
        fernet1 = derive_key(pwd1, b'philosophy_group_salt_l1')
        token_to_name = {
            token: fernet1.decrypt(token.encode()).decode()
            for token in data['_name_tokens'].values()
        }
        sorted_mapping = sorted(token_to_name.items(), key=lambda x: len(x[0]), reverse=True)
        decoded = replace_all(data, sorted_mapping)
        print(f"Level-1 OK — {len(token_to_name)} names restored.")
    except (InvalidToken, Exception) as e:
        raise ValueError(f"Wrong Level-1 password — cannot decrypt names. ({e})")

Level-1 OK — 37 names restored.


## Result

In [ ]:
# `decoded` is the fully (or partially) decrypted dict — use it however you need.
print(f"Messages    : {len(decoded['messages'])}")
print(f"Participants: {len(decoded['participants'])}")
print()
print("First message:")
for msg in decoded['messages'][:1]:
    print(json.dumps(msg, ensure_ascii=False, indent=2))
    print()

Messages    : 44422
Participants: 37

First 3 messages:
{
  "sender_name": "Bế Minh Nhật",
  "timestamp_ms": 1768955883993,
  "content": "Chào cả nhà ^^ mình lập group chat cho buổi Tea & Talk ngày Thứ Bảy 24/1 tới này nha!",
  "is_geoblocked_for_viewer": false,
  "is_unsent_image_by_messenger_kid_parent": false
}

{
  "sender_name": "Bế Minh Nhật",
  "timestamp_ms": 1768955890861,
  "content": "Yoroshiku cả nhà ạ",
  "reactions": [
    {
      "reaction": "❤",
      "actor": "Linh Tran Hoang"
    }
  ],
  "is_geoblocked_for_viewer": false,
  "is_unsent_image_by_messenger_kid_parent": false
}

{
  "sender_name": "Bế Minh Nhật",
  "timestamp_ms": 1768955908508,
  "content": "Nhật đã đặt tên nhóm là MPKEN | \"Lịch sử & Triết học\".",
  "is_geoblocked_for_viewer": false,
  "is_unsent_image_by_messenger_kid_parent": false
}

